Изучение доходности стратегий, постоенной на наборе тех.индикаторов 
Рассчитаваются большое количество индикаторов, из них выбирается случайным образом 3 индикатора из разных категорий и строится стратегия. 
Далее перебираются варианты и ищется наиболее доходная стратегия 

In [25]:
import pandas as pd
import numpy as np
import vectorbt as vbt
import ta
from ta.momentum import RSIIndicator, StochasticOscillator, WilliamsRIndicator
from ta.trend import MACD, ADXIndicator, PSARIndicator, CCIIndicator, IchimokuIndicator, AroonIndicator
from ta.volatility import BollingerBands, AverageTrueRange, KeltnerChannel
from ta.volume import OnBalanceVolumeIndicator, ChaikinMoneyFlowIndicator
import matplotlib.pyplot as plt
from typing import Dict
import itertools
from datetime import datetime


In [26]:

def calculate_indicators(df: pd.DataFrame) -> pd.DataFrame:
    indicators = pd.DataFrame(index=df.index)
    indicators['Close'] = df['Close']
    indicators['Open'] = df['Open']
    indicators['High'] = df['High']
    indicators['Low'] = df['Low']
    indicators['Volume'] = df['Volume']
    
    # 1. SMA, EMA
    for period in [5, 10, 20, 50, 100, 200]:
        indicators[f'SMA_{period}'] = df['Close'].rolling(window=period).mean()
        indicators[f'EMA_{period}'] = df['Close'].ewm(span=period, adjust=False).mean()
    
    # MACD
    macd = MACD(df['Close'])
    indicators['MACD_line'] = macd.macd()
    indicators['MACD_signal'] = macd.macd_signal()
    indicators['MACD_hist'] = macd.macd_diff()
    
    # ADX - Сила тренда
    adx = ADXIndicator(df['High'], df['Low'], df['Close'])
    indicators['ADX'] = adx.adx()
    indicators['DI+'] = adx.adx_pos()
    indicators['DI-'] = adx.adx_neg()
    
    # Parabolic SAR
    psar = PSARIndicator(df['High'], df['Low'], df['Close'])
    indicators['PSAR'] = psar.psar()
    
    # RSI
    for period in [7, 14, 21]:
        rsi = RSIIndicator(df['Close'], window=period)
        indicators[f'RSI_{period}'] = rsi.rsi()
    
    # Stochastic Oscillator
    stoch = StochasticOscillator(df['High'], df['Low'], df['Close'])
    indicators['Stoch_%K'] = stoch.stoch()
    indicators['Stoch_%D'] = stoch.stoch_signal()
    
    # Williams %R
    wr = WilliamsRIndicator(df['High'], df['Low'], df['Close'])
    indicators['Williams_%R'] = wr.williams_r()
    
    # CCI - Commodity Channel Index
    cci = CCIIndicator(df['High'], df['Low'], df['Close'])
    indicators['CCI'] = cci.cci()
    
    # Индикаторы волатильности
    # Bollinger Bands 
    bollinger = BollingerBands(df['Close'])
    indicators['BB_upper'] = bollinger.bollinger_hband()
    indicators['BB_middle'] = bollinger.bollinger_mavg()
    indicators['BB_lower'] = bollinger.bollinger_lband()
    indicators['BB_width'] = (indicators['BB_upper'] - indicators['BB_lower']) / indicators['BB_middle']
    
    # ATR - Average True Range
    atr = AverageTrueRange(df['High'], df['Low'], df['Close'])
    indicators['ATR'] = atr.average_true_range()
    
    # Keltner Channels
    keltner = KeltnerChannel(df['High'], df['Low'], df['Close'])
    indicators['KC_upper'] = keltner.keltner_channel_hband()
    indicators['KC_middle'] = keltner.keltner_channel_mband()
    indicators['KC_lower'] = keltner.keltner_channel_lband()
    
    # 4. Индикаторы объема
    # OBV - On Balance Volume
    obv = OnBalanceVolumeIndicator(df['Close'], df['Volume'])
    indicators['OBV'] = obv.on_balance_volume()
    
    # CMF - Chaikin Money Flow
    cmf = ChaikinMoneyFlowIndicator(df['High'], df['Low'], df['Close'], df['Volume'])
    indicators['CMF'] = cmf.chaikin_money_flow()
    
    # VWAP - Volume Weighted Average Price
    indicators['VWAP'] = (df['Close'] * df['Volume']).cumsum() / df['Volume'].cumsum()
    
    # 5. Дополнительные
    # Ichimoku Cloud
    ichimoku = IchimokuIndicator(df['High'], df['Low'])
    indicators['Ichimoku_A'] = ichimoku.ichimoku_a()
    indicators['Ichimoku_B'] = ichimoku.ichimoku_b()
    
    # Aroon
    aroon = AroonIndicator(high=df['High'], low=df['Low'])
    indicators['Aroon_Up'] = aroon.aroon_up()
    indicators['Aroon_Down'] = aroon.aroon_down()

    for col in indicators.columns:
        if indicators[col].isnull().any():
            indicators[col] = indicators[col].ffill().bfill()
    
    return indicators


In [27]:

def generate_individual_signals(indicators: pd.DataFrame) -> Dict[str, pd.Series]:
    signals = {}
    
    signals['SMA_5_20'] = (indicators['SMA_5'] > indicators['SMA_20']).astype(int)
    signals['SMA_20_50'] = (indicators['SMA_20'] > indicators['SMA_50']).astype(int)
    signals['SMA_50_200'] = (indicators['SMA_50'] > indicators['SMA_200']).astype(int)
    signals['EMA_5_20'] = (indicators['EMA_5'] > indicators['EMA_20']).astype(int)
    signals['EMA_20_50'] = (indicators['EMA_20'] > indicators['EMA_50']).astype(int)
    
    signals['MACD'] = (indicators['MACD_line'] > indicators['MACD_signal']).astype(int)
    
    signals['PSAR'] = (indicators['Close'] > indicators['PSAR']).astype(int)
    
    # ADX with DI Crossover
    signals['ADX_DI'] = ((indicators['ADX'] > 25) & (indicators['DI+'] > indicators['DI-'])).astype(int)
    
    signals['RSI_14_OB_OS'] = pd.Series(0, index=indicators.index)
    signals['RSI_14_OB_OS'][(indicators['RSI_14'] < 30)] = 1  
    signals['RSI_14_OB_OS'][(indicators['RSI_14'] > 70)] = 0   
    
    signals['RSI_14_Trend'] = (indicators['RSI_14'] > 50).astype(int)
    
    # Stochastic Oscillator
    signals['Stochastic_OB_OS'] = pd.Series(0, index=indicators.index)
    signals['Stochastic_OB_OS'][(indicators['Stoch_%K'] < 20) & (indicators['Stoch_%D'] < 20)] = 1  
    signals['Stochastic_OB_OS'][(indicators['Stoch_%K'] > 80) & (indicators['Stoch_%D'] > 80)] = 0  
    
    signals['Stochastic_Cross'] = (indicators['Stoch_%K'] > indicators['Stoch_%D']).astype(int)
    
    # Williams %R
    signals['Williams_R'] = pd.Series(0, index=indicators.index)
    signals['Williams_R'][indicators['Williams_%R'] < -80] = 1  #
    signals['Williams_R'][indicators['Williams_%R'] > -20] = 0 
    
    # CCI
    signals['CCI'] = pd.Series(0, index=indicators.index)
    signals['CCI'][indicators['CCI'] < -100] = 1  
    signals['CCI'][indicators['CCI'] > 100] = 0   
    
    # Bollinger Bands
    signals['Bollinger_Bands'] = pd.Series(0, index=indicators.index)
    signals['Bollinger_Bands'][indicators['Close'] < indicators['BB_lower']] = 1 
    signals['Bollinger_Bands'][indicators['Close'] > indicators['BB_upper']] = 0  
    
    # Keltner Channels
    signals['Keltner_Channels'] = pd.Series(0, index=indicators.index)
    signals['Keltner_Channels'][indicators['Close'] < indicators['KC_lower']] = 1 
    signals['Keltner_Channels'][indicators['Close'] > indicators['KC_upper']] = 0  
    
    # OBV Trend
    obv_ema = indicators['OBV'].ewm(span=20, adjust=False).mean()
    signals['OBV_Trend'] = (indicators['OBV'] > obv_ema).astype(int)
    
    # CMF
    signals['CMF'] = (indicators['CMF'] > 0).astype(int)
    
    # Ichimoku Cloud
    signals['Ichimoku'] = (indicators['Close'] > indicators['Ichimoku_A']).astype(int)
    
    # Aroon
    signals['Aroon'] = (indicators['Aroon_Up'] > indicators['Aroon_Down']).astype(int)
    
    return signals


In [28]:
indicator_types = {
	'trend': ['SMA_5_20', 'SMA_20_50', 'SMA_50_200', 'EMA_5_20', 'EMA_20_50', 'MACD', 'PSAR', 'ADX_DI'],
	'momentum': ['RSI_14_OB_OS', 'RSI_14_Trend', 'Stochastic_OB_OS', 'Stochastic_Cross', 'Williams_R', 'CCI'],
	'volatility': ['Bollinger_Bands', 'Keltner_Channels'],
	'volume': ['OBV_Trend', 'CMF'],
	'other': ['Ichimoku', 'Aroon']
}
categories = [cat for cat, indicators in indicator_types.items() if len(indicators) > 0]
for cats in itertools.combinations(categories, 3):
	print(cats);

('trend', 'momentum', 'volatility')
('trend', 'momentum', 'volume')
('trend', 'momentum', 'other')
('trend', 'volatility', 'volume')
('trend', 'volatility', 'other')
('trend', 'volume', 'other')
('momentum', 'volatility', 'volume')
('momentum', 'volatility', 'other')
('momentum', 'volume', 'other')
('volatility', 'volume', 'other')


In [29]:
# комбинирование индикаторов разных типов

def generate_combined_strategies(signals: Dict[str, pd.Series]) -> Dict[str, pd.Series]:
    
	indicator_types = {
        'trend': ['SMA_5_20', 'SMA_20_50', 'SMA_50_200', 'EMA_5_20', 'EMA_20_50', 'MACD', 'PSAR', 'ADX_DI'],
        'momentum': ['RSI_14_OB_OS', 'RSI_14_Trend', 'Stochastic_OB_OS', 'Stochastic_Cross', 'Williams_R', 'CCI'],
        'volatility': ['Bollinger_Bands', 'Keltner_Channels'],
        'volume': ['OBV_Trend', 'CMF'],
        'other': ['Ichimoku', 'Aroon']
    }

	strategies = {}
	categories = ['trend', 'momentum', 'volatility', 'volume', 'other']

	# перебор разных категорий 
	for cats in itertools.combinations(categories, 3):
		# в цикле перебираем все варианты
		for ind1 in indicator_types[cats[0]]:
			for ind2 in indicator_types[cats[1]]:
				for ind3 in indicator_types[cats[2]]:
					strategy_name = f"{ind1}_{ind2}_{ind3}"
					buy_count = signals[ind1] + signals[ind2] + signals[ind3]
					
					# Сигнал на покупку в стратегии - если сигнал от 2 или 3 индикаторов    
					strategies[strategy_name] = (buy_count >= 2).astype(int)
	
	return strategies


In [30]:
# backtest vectorbt (https://vectorbt.dev/)

def backtest_strategies(strategies: Dict[str, pd.Series], indicators: pd.DataFrame) -> Dict[str, Dict]:
	results = {}
	price_series = indicators['Close']

	init_cash = 1000000
	fees = 0.001
	slippage = 0.001
	stop_loss = 0.05

	strategy_names = list(strategies.keys())

	сount = 0
	for strategy_name in strategy_names:
		signal_series = strategies[strategy_name]
		
		entries = (signal_series == 1) & ((signal_series.shift(1) != 1) | signal_series.shift(1).isna())
		exits = (signal_series == 0) & (signal_series.shift(1) == 1)
		
		portfolio = vbt.Portfolio.from_signals(
			price_series,
			entries,
			exits,
			init_cash=init_cash,
			fees=fees,
			slippage=slippage,
			freq='1d',
			sl_stop=stop_loss
		)
		results[strategy_name] = {
			'total_return': portfolio.total_return(),
			'sharpe_ratio': portfolio.sharpe_ratio()if not np.isnan(portfolio.sharpe_ratio()) else -999,
			'max_drawdown': portfolio.max_drawdown(),
			'win_rate': portfolio.trades.win_rate(), 
			'num_trades': portfolio.trades.count(),
			'profit_factor': portfolio.trades.profit_factor(),
			'portfolio': portfolio
		}
		сount += 1
		if сount % 50 == 0:
			print(f"Tested {сount} strategies...")
			

	print(f"Successfully tested {len(results)} strategies")
	return results


In [31]:
# Сравнение стратегий
def rank_strategies(results: Dict[str, Dict]) -> pd.DataFrame:
	ranking_data = [] 
	for strategy, metrics in results.items():
		if metrics['num_trades'] == 0:
			continue  
			
		ranking_data.append({
			'Strategy': strategy,
			'Total Return': metrics['total_return'],
			'Sharpe Ratio': metrics['sharpe_ratio'],
			'Max Drawdown': metrics['max_drawdown'],
			'Win Rate': metrics['win_rate'],
			'Num Trades': metrics['num_trades'],
			'Profit Factor': metrics['profit_factor'],
			'Indicator1': strategy.split('_')[0],
			'Indicator2': '_'.join(strategy.split('_')[1:-1]), 
			'Indicator3': strategy.split('_')[-1]
		})

	ranking_df = pd.DataFrame(ranking_data)

	# нормализация всех показателей
	for col in ['Total Return', 'Sharpe Ratio', 'Win Rate', 'Profit Factor']:
		min_val = ranking_df[col].min()
		max_val = ranking_df[col].max()
		if max_val - min_val > 0:
			ranking_df[f'{col}_norm'] = (ranking_df[col] - min_val) / (max_val - min_val)
		else:
			ranking_df[f'{col}_norm'] = 0

	# инвертиование макс. просадки, т.к. зависимость обратная
	min_dd = ranking_df['Max Drawdown'].min()
	max_dd = ranking_df['Max Drawdown'].max()
	if max_dd - min_dd > 0:
		ranking_df['Max Drawdown_norm'] = 1 - ((ranking_df['Max Drawdown'] - min_dd) / (max_dd - min_dd))
	else:
		ranking_df['Max Drawdown_norm'] = 1

	# вычисление итогового рейтинга, каждый показатель со своем весом
	if len(ranking_df) > 0:
		ranking_df['Composite Score'] = (
			ranking_df['Total Return_norm'] * 0.35 +
			ranking_df['Sharpe Ratio_norm'] * 0.25 +
			ranking_df['Max Drawdown_norm'] * 0.20 +
			ranking_df['Win Rate_norm'] * 0.10 +
			ranking_df['Profit Factor_norm'] * 0.10
		)

	# Сортировка по рейтингу
	return ranking_df.sort_values('Composite Score', ascending=False)


In [32]:
# 1. Загружаем данные
ticker = 'SBER' #'LKOH'
file_path = "../../data/"+ticker+"/1day.csv"  

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

df = pd.read_csv(file_path)
df.rename(columns={'time': 'Datetime', 'open':'Open','close':'Close', 'high':'High', 'low':'Low', 'volume':'Volume'}, inplace=True)

df.dropna()
len(df)


1834

In [33]:
# 2. Расчет тех индикаторов, возврщается DataFrame с исходными данными+рассчитанными по индикаторам
indicators = calculate_indicators(df)
indicators.columns

Index(['Close', 'Open', 'High', 'Low', 'Volume', 'SMA_5', 'EMA_5', 'SMA_10',
       'EMA_10', 'SMA_20', 'EMA_20', 'SMA_50', 'EMA_50', 'SMA_100', 'EMA_100',
       'SMA_200', 'EMA_200', 'MACD_line', 'MACD_signal', 'MACD_hist', 'ADX',
       'DI+', 'DI-', 'PSAR', 'RSI_7', 'RSI_14', 'RSI_21', 'Stoch_%K',
       'Stoch_%D', 'Williams_%R', 'CCI', 'BB_upper', 'BB_middle', 'BB_lower',
       'BB_width', 'ATR', 'KC_upper', 'KC_middle', 'KC_lower', 'OBV', 'CMF',
       'VWAP', 'Ichimoku_A', 'Ichimoku_B', 'Aroon_Up', 'Aroon_Down'],
      dtype='object')

In [34]:
# 3. Создаем сигналы по каждому индикатору
individual_signals = generate_individual_signals(indicators)


In [35]:
# 4. Создание стратегий
combined_strategies = generate_combined_strategies(individual_signals)


In [36]:
# 6. Backtest 
results = backtest_strategies(combined_strategies, indicators)


Tested 50 strategies...
Tested 100 strategies...
Tested 150 strategies...
Tested 200 strategies...
Tested 250 strategies...
Tested 300 strategies...
Tested 350 strategies...
Tested 400 strategies...
Tested 450 strategies...
Successfully tested 464 strategies


In [37]:
# Сравнение
ranked_strategies = rank_strategies(results)


In [38]:
best_strategy = ranked_strategies.iloc[0]['Strategy']
print("Лучшая стратегия:")
print(best_strategy)


Лучшая стратегия:
SMA_20_50_RSI_14_Trend_Aroon


In [39]:
metrics = results[best_strategy]
print("\nPerformance Metrics:")
print(f"Total Return: {metrics['total_return']:.2%}")
print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
print(f"Max Drawdown: {metrics['max_drawdown']:.2%}")
print(f"Win Rate: {metrics['win_rate']:.2f}")
print(f"Number of Trades: {metrics['num_trades']}")
print(f"Profit Factor: {metrics['profit_factor']:.2f}")


ranked_strategies.to_csv(f"strategy_rankings_{timestamp}.csv")
print(f"Rankings saved to strategy_rankings_{timestamp}.csv")


Performance Metrics:
Total Return: 234.93%
Sharpe Ratio: 1.12
Max Drawdown: -19.72%
Win Rate: 0.46
Number of Trades: 39
Profit Factor: 3.34
Rankings saved to strategy_rankings_20250418_104420.csv


In [40]:
portfolio = results[best_strategy]['portfolio']

# Визуализация результатов бэктеста
fig = portfolio.plot()
fig.update_layout(
    title="Результаты бэктеста лучшей стратегии на техиндикаторах",
    width=1200,
    height=800
)
fig.show()